##### 코랩을 사용할 경우

In [6]:
try:
    # Google Drive를 Colab(/content/drive)에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec05
    %ls
except ImportError:
    pass

##### 이전 노트북 실행

In [7]:
%%capture
get_ipython().run_line_magic("run", "01_data_preprocessing.ipynb")
get_ipython().run_line_magic("run", "02_model_definition.ipynb")

##### 임포트

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

##### Device 설정

In [9]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### 옵티마이저 생성

In [10]:
# 손실 함수(CrossEntropyLoss) 생성
loss_fn = nn.CrossEntropyLoss()

# Adam 옵티마이저(Optimizer) 생성
# lr=0.001: 학습률 (모든 파라미터(가중치, 편향)를 얼마나 크게 변경할지)
optimizer = optim.Adam(model.parameters(), lr=0.001)

##### 학습하기 (조기 종료 포함)

In [11]:
# 학습 에포크 및 배치 크기 설정
epochs     = 200
batch_size = 16

# 조기 종료(Early Stopping) 설정 #####################################
# patience: 검증 손실이 개선되지 않아도 허용할 에포크 수
# patience_counter: 개선 없이 누적된 에포크 수
# best_val_loss: 현재까지 가장 낮은 검증 손실 (초기값: 무한대)
# best_model_state: 검증 손실이 가장 낮았을 때의 모델 파라미터 복사본
patience         = 10
patience_counter = 0
best_val_loss    = float('inf') # 양의 무한대 (최소값을 찾기 위해 초기값을 크게 설정)
best_model_state = None
####################################################################

# 에포크 반복
for epoch in range(epochs):
    # 훈련 단계 -----------------------------------------------------
    # 학습 모드로 설정: 파라미터 업데이트 및 드롭아웃 활성화
    model.train()

    # 훈련 손실과 정확도 누적 변수 초기화
    batch_loss_sum   = 0
    batch_correct_sum = 0

    for i in range(0, len(train_x_tensor), batch_size):
        batch_x_tensor = train_x_tensor[i:i + batch_size]
        batch_y_tensor = train_y_tensor[i:i + batch_size]

        # 1. 그래디언트 초기화
        optimizer.zero_grad()

        # 2. 모델 출력 얻기
        batch_train_outputs = model(batch_x_tensor)

        # 3. 손실 계산
        loss = loss_fn(batch_train_outputs, batch_y_tensor)

        # 4. 역전파: 그래디언트 계산
        loss.backward()

        # 5. 가중치 업데이트
        optimizer.step()

        # 각 배치 손실의 합산
        batch_loss_sum  += loss.item()

        # 각 배치에서 맞춘 개수 합산: sum(): True(=1) 개수 합산
        batch_correct_sum += (batch_train_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

    # 훈련 평균 손실: 각 배치 손실의 합산 / 배치 개수
    epoch_loss = batch_loss_sum / (len(train_x_tensor) / batch_size)

    # 훈련 정확도: 맞춘 개수 / 전체 개수 * 100
    epoch_acc = batch_correct_sum / len(train_x_tensor) * 100

    # 검증 단계 -----------------------------------------------------
    # 평가 모드로 설정: 파라미터 업데이트 및 드롭아웃 비활성화
    model.eval()

    # 검증 손실과 정확도 누적 변수 초기화
    val_loss    = 0
    val_correct = 0

    # 역전파/기울기 계산없이 검증셋으로 예측 수행
    with torch.inference_mode():  # torch.no_grad():
        # 검증셋 예측
        val_outputs  = model(val_x_tensor)
        # 검증 손실
        val_loss = loss_fn(val_outputs, val_y_tensor).item()
        # 검증 정확도
        val_correct = (val_outputs.argmax(dim=1) == val_y_tensor).sum().item()
        val_acc = val_correct / len(val_x_tensor) * 100

    # 출력하기 -----------------------------------------------------
    # 에포크마다 훈련·검증 지표를 나란히 출력하여 과적합 여부 확인
    print(f"Epoch [{epoch+1:3d}/{epochs}]  "
            f"훈련 손실: {epoch_loss:.4f}  훈련 정확도: {epoch_acc:5.1f}%  "
            f"검증 손실: {val_loss:.4f}  검증 정확도: {val_acc:5.1f}%")

    # 조기 종료 판단 ------------------------------------------------
    if val_loss < best_val_loss:
        # 검증 손실이 개선된 경우: 최적 상태 갱신
        best_val_loss    = val_loss
        patience_counter = 0
        # 현재 에포크의 모델 파라미터를 별도 메모리에 복사
        best_model_state = copy.deepcopy(model.state_dict())
        print("  ✓ 최적 모델 저장")
    else:
        # 검증 손실이 개선되지 않은 경우: 카운터 증가
        patience_counter += 1
        print(f"  (개선 없음 {patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"\n조기 종료: {patience} 에포크 동안 검증 손실 개선 없음")
            break

# 가장 성능이 좋았던 에포크의 가중치로 모델 복원
model.load_state_dict(best_model_state)
print(f"학습 완료  (최적 검증 손실: {best_val_loss:.4f})")

Epoch [  1/200]  훈련 손실: 1.0652  훈련 정확도:  52.8%  검증 손실: 1.0189  검증 정확도:  83.3%
  ✓ 최적 모델 저장
Epoch [  2/200]  훈련 손실: 1.0015  훈련 정확도:  69.7%  검증 손실: 0.9517  검증 정확도:  94.4%
  ✓ 최적 모델 저장
Epoch [  3/200]  훈련 손실: 0.9225  훈련 정확도:  80.3%  검증 손실: 0.8714  검증 정확도:  94.4%
  ✓ 최적 모델 저장
Epoch [  4/200]  훈련 손실: 0.8606  훈련 정확도:  85.2%  검증 손실: 0.7791  검증 정확도:  94.4%
  ✓ 최적 모델 저장
Epoch [  5/200]  훈련 손실: 0.7358  훈련 정확도:  95.1%  검증 손실: 0.6720  검증 정확도:  94.4%
  ✓ 최적 모델 저장
Epoch [  6/200]  훈련 손실: 0.6467  훈련 정확도:  91.5%  검증 손실: 0.5577  검증 정확도:  94.4%
  ✓ 최적 모델 저장
Epoch [  7/200]  훈련 손실: 0.5252  훈련 정확도:  95.1%  검증 손실: 0.4432  검증 정확도:  94.4%
  ✓ 최적 모델 저장
Epoch [  8/200]  훈련 손실: 0.4178  훈련 정확도:  95.8%  검증 손실: 0.3404  검증 정확도: 100.0%
  ✓ 최적 모델 저장
Epoch [  9/200]  훈련 손실: 0.3462  훈련 정확도:  95.1%  검증 손실: 0.2567  검증 정확도: 100.0%
  ✓ 최적 모델 저장
Epoch [ 10/200]  훈련 손실: 0.2554  훈련 정확도:  99.3%  검증 손실: 0.1966  검증 정확도: 100.0%
  ✓ 최적 모델 저장
Epoch [ 11/200]  훈련 손실: 0.1972  훈련 정확도:  97.9%  검증 손실: 0.1583  검증 정확도: 100.0%
  ✓ 최적 모델 저장